In [ ]:
from io import BytesIO
import os
import re
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib.utils import ImageReader
from PIL import Image
import numpy as np
from google.generativeai import GenerativeModel

# Convert a frame (numpy array) to PIL Image
def frame_to_pil(frame):
    if frame.dtype != np.uint8:
        frame = (frame * 255).clip(0, 255).astype(np.uint8)
    return Image.fromarray(frame)

# Generate PDF from selected keyframes
def generate_notes_pdf(frames, keyframes, outdir="output", filename="lecture_notes.pdf"):
    os.makedirs(outdir, exist_ok=True)
    pdf_path = os.path.join(outdir, filename)

    # Step 1: Use Gemini to select relevant frames
    model = GenerativeModel("gemini-2.0-flash")
    context_message = (
        "You are a lecture summarization vision model.\n"
        "Select keyframes from a lecture video that contain full, readable educational content suitable for study notes.\n"
        "Include frames that have blackboard, slides, formulas, diagrams, charts, code, or readable text.\n"
        "Exclude frames with people, talking heads, intros/outros, blank or blurry frames.\n"
        "Return strictly a JSON array of frame indices relative to the keyframes list, e.g. [0, 3, 5]."
        # "try to take frames that vary from each other in terms of the content in the frame ignore small changes.\n"
    )

    message_parts = [context_message]
    for kf in keyframes[:80]:  
        pil_img = frame_to_pil(frames[kf['frame_index']])
        buffered = BytesIO()
        pil_img.save(buffered, format="PNG")
        img_bytes = buffered.getvalue()
        message_parts.append({"mime_type": "image/png", "data": img_bytes})

    print("🔍 Sending frames to Gemini for filtering...")
    response = model.generate_content(message_parts)
    print("Gemini selection response:\n", response.text)

    # Extract indices
    indices = [int(i) for i in re.findall(r"\d+", response.text)]
    if not indices:
        print("⚠️ No relevant frames detected. PDF will be empty.")
        return pdf_path

    print(f"✅ Selected frames: {indices}")

    # Step 2: Create PDF
    c = canvas.Canvas(pdf_path, pagesize=A4)
    width, height = A4
    margin = 40

    for i in indices:
        kf = keyframes[i]
        pil_img = frame_to_pil(frames[kf['frame_index']])
        img_buffer = BytesIO()
        pil_img.save(img_buffer, format='PNG')
        img_buffer.seek(0)
        img_reader = ImageReader(img_buffer)

        img_width, img_height = pil_img.size
        scale = min((width - 2*margin) / img_width, (height - 2*margin) / img_height)
        new_width = img_width * scale
        new_height = img_height * scale
        x = (width - new_width) / 2
        y = (height - new_height) / 2

        c.drawImage(img_reader, x, y, new_width, new_height)
        c.showPage()

    c.save()
    print(f"📘 PDF created successfully at: {pdf_path}")
    return pdf_path
